# Chaldal.com Web Scraper
## A Simple Guide to Scraping Product Data from Chaldal

This notebook shows you how to scrape any product page from Chaldal.com and export the data to JSON, CSV, or Excel.

**What you'll learn:**
- How to fetch pages from websites
- How to extract product information (name, price, quantity)
- How to handle both regular and discounted prices
- How to save data in different formats (JSON, CSV, Excel)

## 1. Import Required Libraries

We need a few libraries to scrape websites:
- `requests` - to fetch web pages
- `BeautifulSoup` - to extract data from HTML
- `json`, `csv` - to save data in different formats
- `pandas` - to organize data in tables

In [8]:
import requests
from bs4 import BeautifulSoup
import json
import csv
import pandas as pd
import time
from datetime import datetime

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


## 2. Define the Scraper Class

A scraper class is like a robot that:
1. Visits a website
2. Reads the page content
3. Finds product information
4. Stores it in a structured format

This class handles all the technical details so you just need to provide a URL.

In [9]:
class ChaldalScraper:
    def __init__(self):
        # Set up the header so websites think we're a real browser
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        self.base_url = "https://chaldal.com"
        
    def fetch_page(self, url):
        """Visit a website and get the HTML content"""
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
            response.raise_for_status()
            return BeautifulSoup(response.content, 'html.parser')
        except requests.RequestException as e:
            print(f"Error fetching {url}: {e}")
            return None
    
    def extract_products(self, soup):
        """Find all products on the page and extract their information"""
        products = []
        product_containers = soup.find_all('div', class_='productV2Catalog')
        
        for container in product_containers:
            product = self._extract_single_product(container)
            if product:
                products.append(product)
        
        return products
    
    def _extract_single_product(self, container):
        """Extract details from one product container"""
        try:
            product = {}
            
            # Product name
            name_elem = container.find('p', class_='nameTextWithEllipsis')
            product['name'] = name_elem.get_text(strip=True) if name_elem else 'N/A'
            
            # Price (handles both discounted and regular pricing)
            discounted_price_div = container.find('div', class_='productV2discountedPrice')
            if discounted_price_div:
                # Get discounted price
                disc_span = discounted_price_div.find('span')
                product['discounted_price'] = f"৳ {disc_span.get_text(strip=True)}" if disc_span else 'N/A'
                
                # Get original price
                original_div = discounted_price_div.find('div', class_='price')
                if original_div:
                    orig_span = original_div.find('span')
                    product['original_price'] = f"৳ {orig_span.get_text(strip=True)}" if orig_span else 'N/A'
                else:
                    product['original_price'] = 'N/A'
                product['price'] = product['discounted_price']
            else:
                # Regular price (no discount)
                price_div = container.find('div', class_='price')
                if price_div:
                    span = price_div.find('span')
                    price = f"৳ {span.get_text(strip=True)}" if span else 'N/A'
                else:
                    price = 'N/A'
                product['price'] = price
                product['discounted_price'] = 'N/A'
                product['original_price'] = 'N/A'
            
            # Quantity/Size
            subtext = container.find('div', class_='subText')
            if subtext:
                qty_elem = subtext.find('span')
                product['quantity'] = qty_elem.get_text(strip=True) if qty_elem else 'N/A'
            else:
                product['quantity'] = 'N/A'
            
            # Delivery time
            delivery = container.find('div', class_='deliveryTimeText')
            if delivery:
                deliv_span = delivery.find('span')
                product['delivery_time'] = deliv_span.get_text(strip=True) if deliv_span else 'N/A'
            else:
                product['delivery_time'] = 'N/A'
            
            # Image URL
            img = container.find('img')
            product['image_url'] = img.get('src') if img else 'N/A'
            
            return product if product.get('name') != 'N/A' else None
        
        except Exception as e:
            return None
    
    def scrape_url(self, url):
        """Main method: scrape a Chaldal page"""
        print(f"🔍 Fetching: {url}")
        soup = self.fetch_page(url)
        
        if not soup:
            return None
        
        print("📝 Extracting product data...")
        products = self.extract_products(soup)
        
        result = {
            'url': url,
            'timestamp': datetime.now().isoformat(),
            'total_products': len(products),
            'products': products
        }
        
        print(f"✓ Found {len(products)} products")
        return result

print("✓ Scraper class created successfully!")

✓ Scraper class created successfully!


## 3. Initialize the Scraper

Now let's create an instance of our scraper. Think of it like turning on the robot.

In [10]:
# Create the scraper
scraper = ChaldalScraper()
print("✓ Scraper is ready to use!")

✓ Scraper is ready to use!


## 4. Save Data to Different Formats

Let's create functions to save the scraped data in different formats (JSON, CSV, Excel).

In [11]:
def save_as_json(data, filename):
    """Save data as JSON file"""
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"✓ Saved to: {filename}")

def save_as_csv(data, filename):
    """Save data as CSV file"""
    products = data.get('products', [])
    if not products:
        print("No products to save")
        return
    
    # Get all column names
    fieldnames = set()
    for product in products:
        fieldnames.update(product.keys())
    fieldnames = sorted(list(fieldnames))
    
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(products)
    print(f"✓ Saved to: {filename}")

def save_as_excel(data, filename):
    """Save data as Excel file with formatting"""
    try:
        import openpyxl
        from openpyxl.styles import Font, PatternFill, Alignment
    except ImportError:
        print("Install openpyxl: pip install openpyxl")
        return
    
    products = data.get('products', [])
    df = pd.DataFrame(products)
    
    # Save with Excel formatting
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Products', index=False)
        
        # Style the header row
        workbook = writer.book
        worksheet = writer.sheets['Products']
        header_fill = PatternFill(start_color="7533CB", end_color="7533CB", fill_type="solid")
        header_font = Font(bold=True, color="FFFFFF")
        
        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
        
        # Auto-adjust column widths
        for column in worksheet.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    max_length = max(max_length, len(str(cell.value)))
                except:
                    pass
            worksheet.column_dimensions[column_letter].width = min(max_length + 2, 50)
    
    print(f"✓ Saved to: {filename}")

def show_preview(data, limit=5):
    """Show a preview of the scraped data"""
    df = pd.DataFrame(data['products'])
    print(f"\n📊 Total products: {data['total_products']}")
    print(f"\nFirst {limit} products:\n")
    print(df[['name', 'price', 'quantity', 'delivery_time']].head(limit).to_string())

print("✓ Save functions ready!")

✓ Save functions ready!


## 5. Example: Scrape a Chaldal Product Page

Now let's put it all together! You can change the URL to any Chaldal product page.

**Example URLs you can try:**
- https://chaldal.com/coffees
- https://chaldal.com/fresh-vegetable
- https://chaldal.com/beverages
- https://chaldal.com/food
- https://chaldal.com/cooking-sauce

In [12]:
# ============== CHANGE THIS URL TO SCRAPE DIFFERENT PAGES ==============
# url = "https://chaldal.com/coffees"
# Try other URLs like:
url = "https://chaldal.com/fresh-vegetable"
# url = "https://chaldal.com/cooking-sauce"
# url = "https://chaldal.com/beverages"

print(f"\n{'='*60}")
print(f"Scraping: {url}")
print(f"{'='*60}\n")

# Scrape the page
data = scraper.scrape_url(url)

# If successful, show preview and save
if data:
    # Show preview
    show_preview(data)
    
    # Choose output format and filename
    # Options: json, csv, or excel
    
    # Save as JSON
    save_as_json(data, "products_data.json")
    
    # Save as CSV
    save_as_csv(data, "products_data.csv")
    
    # Save as Excel
    save_as_excel(data, "products_data.xlsx")
    
    print(f"\n{'='*60}")
    print(f"✓ Successfully scraped and saved {data['total_products']} products!")
    print(f"{'='*60}")
else:
    print("❌ Failed to scrape the page")


Scraping: https://chaldal.com/fresh-vegetable

🔍 Fetching: https://chaldal.com/fresh-vegetable
📝 Extracting product data...
✓ Found 50 products

📊 Total products: 50

First 5 products:

                    name price quantity delivery_time
0  Fulkopi (Cauliflower)  ৳ 49     each          1 hr
1      Flat Bean (Sheem)  ৳ 29   500 gm          1 hr
2    Badhakopi (Cabbage)  ৳ 45     each          1 hr
3           Onion Flower  ৳ 19   250 gm          1 hr
4               Broccoli  ৳ 65     each          1 hr
✓ Saved to: products_data.json
✓ Saved to: products_data.csv
✓ Saved to: products_data.xlsx

✓ Successfully scraped and saved 50 products!


## 6. Analyze the Scraped Data (Optional)

After scraping, you can use Pandas to analyze the data in various ways.

In [13]:
if data:
    # Convert to DataFrame for analysis
    df = pd.DataFrame(data['products'])
    
    print("\n📊 Data Analysis:\n")
    
    # Show basic info
    print(f"Total columns: {len(df.columns)}")
    print(f"Total rows: {len(df)}")
    
    # Show all columns
    print(f"\nAvailable columns: {list(df.columns)}")
    
    # Show full data
    print("\n✓ Complete Dataset:")
    print(df.to_string())
    
    # Show products with discounts
    df_with_discounts = df[df['discounted_price'] != 'N/A']
    if len(df_with_discounts) > 0:
        print(f"\n💰 Products with discounts: {len(df_with_discounts)}")
        print(df_with_discounts[['name', 'discounted_price', 'original_price']].head(10).to_string())


📊 Data Analysis:

Total columns: 7
Total rows: 50

Available columns: ['name', 'discounted_price', 'original_price', 'price', 'quantity', 'delivery_time', 'image_url']

✓ Complete Dataset:
                                              name discounted_price original_price  price  quantity delivery_time                                                                                                                                                                      image_url
0                            Fulkopi (Cauliflower)             ৳ 49           ৳ 59   ৳ 49      each          1 hr                   https://chaldn.com/_mpimage/fulkopi-cauliflower-1-pcs?src=https%3A%2F%2Feggyolk.chaldal.com%2Fapi%2FPicture%2FRaw%3FpictureId%3D46971&q=best&v=1&m=400&m=400
1                                Flat Bean (Sheem)              N/A            N/A   ৳ 29    500 gm          1 hr                            https://chaldn.com/_mpimage/flat-bean-sheem-500-gm?src=https%3A%2F%2Feggyolk.chaldal.com%2F

## 7. Tips & Troubleshooting

### 🎯 Quick Tips:
- **Change the URL** to scrape different product categories
- **Save in multiple formats** for flexibility (JSON for APIs, CSV for Excel, XLSX for reports)
- **Check the preview** to make sure data looks correct before saving
- **Extracted fields**: name, price, discounted_price, original_price, quantity, delivery_time, image_url

### ❓ Common Issues:

**Q: Script doesn't find products**
- The website structure may have changed
- Try a different URL to test if the scraper works
- Check that you have internet connection

**Q: Excel file not created**
- Install openpyxl: `pip install openpyxl`
- Or just use CSV or JSON format instead

**Q: Getting permission error**
- You might not have write access to the folder
- Move to a different folder where you have permissions
- Or run Jupyter with administrator privileges

**Q: Products showing as "N/A"**
- The website may have a different page structure
- Try refreshing the page in your browser
- The product might be out of stock or removed

## 8. Quick Reference: Popular Chaldal Categories

**Popular Product Categories:**
- Coffees: `https://chaldal.com/coffees`
- Teas: `https://chaldal.com/teas`
- Fresh Vegetables: `https://chaldal.com/fresh-vegetables`
- Fresh Fruits: `https://chaldal.com/fresh-fruits`
- Milk & Dairy: `https://chaldal.com/milk-dairy`
- Spices: `https://chaldal.com/spices`
- Rice & Grains: `https://chaldal.com/rice-grains`

**Pro Tips:**
- Copy any category URL from the website
- This scraper works with any Chaldal category page
- Some categories may have pagination (50+ products) - the scraper captures what's on the current page
- All product fields are saved automatically